In [1]:
# =========================
# define rooms and items
# =========================

# Objetos por sala (uno correcto y dos distractores)
hairfork = {"name": "hairfork", "type": "object"}   # correcto en jail
book     = {"name": "book",     "type": "object"}
bed      = {"name": "bed",      "type": "object"}

nursing_key = {"name": "key for nursing", "type": "object"}  # correcto en cantine
dish        = {"name": "dish",            "type": "object"}
sandwich    = {"name": "sandwich",        "type": "object"}

gown          = {"name": "gown",             "type": "object"}  # correcto en nursing
medical_kit   = {"name": "medical kit",      "type": "object"}
doctors_glass = {"name": "doctor's glasses", "type": "object"}

go_right_cantine = {"name": "go right (cantine)", "type": "object"}   # correcto en hallway_1
go_left_nursing  = {"name": "go left (nursing - locked)", "type": "object"}
go_to_guards     = {"name": "go to guards (bad idea)", "type": "object"}

go_right_yard   = {"name": "go right (yard)", "type": "object"}       # correcto en hallway_2
go_left_guards  = {"name": "go left (guards)", "type": "object"}
go_back_nursing = {"name": "go back (nursing)", "type": "object"}

ladder = {"name": "ladder", "type": "object"}  # correcto en yard
ball   = {"name": "ball",   "type": "object"}
rake   = {"name": "rake",   "type": "object"}

# Rooms
jail     = {"name": "jail",     "type": "room"}
hallway  = {"name": "hallway",  "type": "room"}  # usaremos el mismo "name" en dos fases
cantine  = {"name": "cantine",  "type": "room"}
nursing  = {"name": "nursing",  "type": "room"}
yard     = {"name": "yard",     "type": "room"}
outside  = {"name": "outside"}  # destino final

all_rooms = [jail, hallway, cantine, nursing, yard, outside]
all_doors = []  # no usamos puertas explícitas en esta variante

# =========================
# define which items/rooms are related
# =========================
object_relations = {
    "jail":    [hairfork, book, bed],
    "hallway": [],  # se rellena dinámicamente según fase
    "cantine": [nursing_key, dish, sandwich],
    "nursing": [gown, medical_kit, doctors_glass],
    "yard":    [ladder, ball, rake],
    "outside": []
}

HALLWAY_1_OPTIONS = [go_right_cantine, go_left_nursing, go_to_guards]
HALLWAY_2_OPTIONS = [go_right_yard,   go_left_guards,  go_back_nursing]

CORRECT_BY_PHASE = {
    "jail":       "hairfork",
    "hallway_1":  "go right (cantine)",
    "cantine":    "key for nursing",
    "nursing":    "gown",
    "hallway_2":  "go right (yard)",
    "yard":       "ladder",
}

# =========================
# game state (como el ejemplo)
# =========================
INIT_GAME_STATE = {
    "current_room": jail,
    "keys_collected": [],
    "target_room": outside,
    "phase": "jail",        # jail -> hallway_1 -> cantine -> nursing -> hallway_2 -> yard -> outside
}

def linebreak():
    print("\n")

# =========================
# helpers
# =========================
def norm(s: str) -> str:
    # normaliza: minúsculas, sin espacios/extremos ni comillas
    return s.strip().strip('"').strip("'").lower()

def list_option_names(options):
    return [opt["name"] for opt in options]

def prompt_choice(options):
    """
    Pide una elección:
      - acepta 'examine <texto>'
      - acepta solo el nombre (p. ej. hairfork)
      - acepta 1/2/3 (según listado)
    Devuelve el dict del objeto elegido o None si no hay match.
    """
    names = list_option_names(options)
    # mostrar opciones numeradas
    print("Choose one to use:")
    for i, n in enumerate(names, 1):
        print(f"  {i}) {n}")
    raw = input("> ").strip()
    lo = norm(raw)

    # números 1..len
    if lo.isdigit():
        idx = int(lo) - 1
        if 0 <= idx < len(options):
            return options[idx]
        return None

    # 'examine X' o 'examinar X'
    if lo.startswith("examine ") or lo.startswith("examinar "):
        after = raw.split(" ", 1)[1] if " " in raw else ""
        lo = norm(after)

    # match por nombre (case-insensitive, sin comillas)
    for opt in options:
        if norm(opt["name"]) == lo:
            return opt
    return None

# =========================
# core loop (siguiendo tu plantilla)
# =========================
def start_game():
    print("You wake up in a jail cell. In each room, pick the right item to progress.")
    play_room(game_state["current_room"])

def play_room(room):
    game_state["current_room"] = room

    # ¿objetivo alcanzado?
    if game_state["current_room"] == game_state["target_room"]:
        print("Congrats! You escaped!")
        return

    # Determinar opciones según fase
    phase = game_state["phase"]
    if phase == "jail":
        options = object_relations["jail"]
    elif phase == "hallway_1":
        options = HALLWAY_1_OPTIONS
    elif phase == "cantine":
        options = object_relations["cantine"]
    elif phase == "nursing":
        options = object_relations["nursing"]
    elif phase == "hallway_2":
        options = HALLWAY_2_OPTIONS
    elif phase == "yard":
        options = object_relations["yard"]
    else:
        options = []

    print("You are now in " + room["name"])
    ask_until_correct(options)

def ask_until_correct(options):
    correct_name = CORRECT_BY_PHASE[game_state["phase"]]

    while True:
        picked = prompt_choice(options)
        if picked is None:
            print("Not found. Try again.")
            continue
        if norm(picked["name"]) == norm(correct_name):
            advance()
            return
        else:
            print("Not the right item. Try again.")

def advance():
    # cambia fase y sala según tu historia
    phase = game_state["phase"]
    if phase == "jail":
        game_state["phase"] = "hallway_1"
        temp_hallway = {"name": "hallway", "type": "room"}
        play_room(temp_hallway)
    elif phase == "hallway_1":
        game_state["phase"] = "cantine"
        play_room(cantine)
    elif phase == "cantine":
        game_state["phase"] = "nursing"
        play_room(nursing)
    elif phase == "nursing":
        game_state["phase"] = "hallway_2"
        temp_hallway = {"name": "hallway", "type": "room"}
        play_room(temp_hallway)
    elif phase == "hallway_2":
        game_state["phase"] = "yard"
        play_room(yard)
    elif phase == "yard":
        game_state["phase"] = "outside"
        play_room(outside)

# =========================
# start
# =========================
game_state = INIT_GAME_STATE.copy()
start_game()


You wake up in a jail cell. In each room, pick the right item to progress.
You are now in jail
Choose one to use:
  1) hairfork
  2) book
  3) bed
You are now in hallway
Choose one to use:
  1) go right (cantine)
  2) go left (nursing - locked)
  3) go to guards (bad idea)
Not found. Try again.
Choose one to use:
  1) go right (cantine)
  2) go left (nursing - locked)
  3) go to guards (bad idea)
You are now in cantine
Choose one to use:
  1) key for nursing
  2) dish
  3) sandwich
You are now in nursing
Choose one to use:
  1) gown
  2) medical kit
  3) doctor's glasses
You are now in hallway
Choose one to use:
  1) go right (yard)
  2) go left (guards)
  3) go back (nursing)
You are now in yard
Choose one to use:
  1) ladder
  2) ball
  3) rake
Congrats! You escaped!
